In [ ]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 1s (6,948 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package fonts-nanum.
(Reading database ... 121703 files and dire

# 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rc('font', family='NanumBarunGothic')

!pip install tspoon
import tspoon as tsp

In [ ]:
!pip install xmltodict
import pandas as pd

import requests
import xml.etree.ElementTree as ET
import xml.dom.minidom
import xmltodict

import json
from urllib.request import urlopen

def getECOS(topic,period,beg,end,item1='?',item2='?',item3='?',item4='?', df=None):
    key = ''
    url = 'https://ecos.bok.or.kr/api/StatisticSearch/'+'OIT9OYFOZDZWLHSBQJWU'+'/xml/kr/'+str(1)+'/'+str(50000)+'/'+topic+'/'+period+'/'+beg+'/'+end \
            +'/'+item1+'/'+item2+'/'+item3+'/'+item4
    # print(url)
    ## call OpenAPI URL
    response = requests.get(url)

    ## get API return value upon the http request
    if response.status_code == 200:
        try:
            contents = response.text
            ecosRoot = ET.fromstring(contents)

            # error check
            if ecosRoot[0].text[:4] in ("INFO","ERRO"):
                print(ecosRoot[0].text + " : " + ecosRoot[1].text)

            else:
                #print(contents)
                #dom = xml.dom.minidom.parse(xml_fname)
                dom = xml.dom.minidom.parseString(contents)
                pretty_xml_as_string = dom.toprettyxml(indent=" ")
                # print(pretty_xml_as_string)
                dic = xmltodict.parse(pretty_xml_as_string)

                n = int(dic['StatisticSearch']['list_total_count']['#text'])
                df_ecos = pd.DataFrame(index = [dic['StatisticSearch']['row'][i]['TIME'] for i in range(n)])
                df_ecos[dic['StatisticSearch']['row'][0]['ITEM_NAME1']] = [float(dic['StatisticSearch']['row'][i]['DATA_VALUE']) for i in range(n)]

                if type(df)==type(pd.DataFrame()):
                    df_ecos = df.merge(df_ecos, left_index=True, right_index=True, how='left')

                return df_ecos

        except Exception as e:
            print(str(e))


def getKOSIS(table,period,beg,end,item, orgId='101', obj1='',obj2='',obj3='',obj4='',obj5='',obj6='',obj7='',obj8='', title=None, title_no='0', debug=False):
     # Korean Statistics
    key = 'MzhiMmQwZTYyMWZiNGM1ZGFkZDJmZTI2OTE1OGU2NzQ='
    url = 'https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey='+key \
            +'&orgId='+orgId\
            +'&tblId='+table \
            +'&prdSe='+period \
            +'&startPrdDe='+beg \
            +'&endPrdDe='+end \
            +'&itmId='+item \
            +'&objL1='+obj1 \
            +'&objL2='+obj2 \
            +'&objL3='+obj3 \
            +'&objL4='+obj4 \
            +'&objL5='+obj5 \
            +'&objL6='+obj6 \
            +'&objL7='+obj7 \
            +'&objL8='+obj8 \
            +'&format=json&jsonVD=Y&outputFields=ITM_NM+PRD_DE+DT'
    res = requests.get(url)
    py_json = res.json()

    data = []
    for v in py_json:
        value = []
        value.append(v['PRD_DE'])

        # ✅ DT 값이 '-' 또는 ''일 때 NaN으로 처리
        try:
            value.append(float(v['DT']))
        except (ValueError, KeyError):
            value.append(np.nan)

        data.append(value)

    df = pd.DataFrame(data, columns=['PRD_DE', 'DT'])

    if debug:
        print(df.head())

    return df


    #get json data from url
    with urlopen(url) as url_:
        json_file = url_.read()

    py_json = json.loads(json_file.decode('utf-8'))


    #data transformation
    data = []

    for i, v in enumerate(py_json):
        value = []
        value.append(v['PRD_DE'])
        value.append(float(v['DT']))

        data.append(value)

    if title is not None:
        title = str(title)
    elif title_no=='0':
        title = v['ITM_NM_ENG']
    else:
        title = v['C'+title_no+'_NM_ENG']

    df_kosis = pd.DataFrame({v['PRD_SE']:[x[0] for x in data], title:[x[1] for x in data]}).set_index(v['PRD_SE'])

    if debug:
        return v

    return df_kosis

In [ ]:
import pandas as pd
import numpy as np
import os
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/', force_remount=True)

Mounted at /content/gdrive/


In [ ]:
# change directory to where you have your data file!!
%cd gdrive/MyDrive/GDP/데이터

/content/gdrive/MyDrive/GDP/데이터


## GDP

In [ ]:
gdp = getECOS('200Y104', 'Q', '2014Q1', '2025Q2','1107').reset_index()
gdp
gdp = gdp.rename(columns={'운수업':'gdp'})
gdp

,index,gdp
0,2014Q1,16590.6
1,2014Q2,16624.8
2,2014Q3,16808.5
3,2014Q4,17112.4
4,2015Q1,16893.8
5,2015Q2,17061.1
6,2015Q3,17504.2
7,2015Q4,17520.5
8,2016Q1,17465.3
9,2016Q2,17655.0


##PPI

In [ ]:
ppi = getECOS('404Y014','Q','2014Q1','2025Q2','501AA').reset_index()
ppi

,index,운송서비스
0,2014Q1,94.67
1,2014Q2,95.01
2,2014Q3,95.08
3,2014Q4,94.60
4,2015Q1,93.75
5,2015Q2,94.21
6,2015Q3,94.64
7,2015Q4,94.47
8,2016Q1,94.59
9,2016Q2,94.43


In [ ]:
ppi = ppi.rename(columns={'운송서비스':'ppi'})
ppi

,index,ppi
0,2014Q1,94.67
1,2014Q2,95.01
2,2014Q3,95.08
3,2014Q4,94.60
4,2015Q1,93.75
5,2015Q2,94.21
6,2015Q3,94.64
7,2015Q4,94.47
8,2016Q1,94.59
9,2016Q2,94.43


##BSI

In [ ]:
업황실적BSI = getECOS('512Y007','M','201401','202506','AA','H4900').reset_index()
매출실적BSI = getECOS('512Y007','M','201401','202506','AB','H4900').reset_index()
채산성실적BSI = getECOS('512Y007','M','201401','202506','AE','H4900').reset_index()
자금사정실적BSI = getECOS('512Y007','M','201401','202506','AO','H4900').reset_index()
인력사정실적BSI = getECOS('512Y007','M','201401','202506','AJ','H4900').reset_index()


업황실적BSI = 업황실적BSI.rename(columns={'업황실적BSI 1)':'업황실적BSI'})
매출실적BSI = 매출실적BSI.rename(columns={'매출실적BSI 2)':'매출실적BSI'})
채산성실적BSI = 채산성실적BSI.rename(columns={'채산성실적BSI 6)':'채산성실적BSI'})
자금사정실적BSI = 자금사정실적BSI.rename(columns={'자금사정실적BSI 6)':'자금사정실적BSI'})
인력사정실적BSI = 인력사정실적BSI.rename(columns={'인력사정실적BSI 3)':'인력사정실적BSI'})

In [ ]:
from functools import reduce

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [업황실적BSI, 매출실적BSI, 채산성실적BSI, 자금사정실적BSI, 인력사정실적BSI]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
df_merged = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
df_merged

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI
0,201401,68.0,84.0,83.0,91.0,88.0
1,201402,77.0,91.0,87.0,89.0,91.0
2,201403,72.0,81.0,86.0,84.0,89.0
3,201404,81.0,94.0,92.0,91.0,86.0
4,201405,80.0,93.0,91.0,89.0,95.0
...,...,...,...,...,...,...
133,202502,73.0,75.0,73.0,78.0,79.0
134,202503,74.0,81.0,85.0,82.0,82.0
135,202504,72.0,76.0,82.0,77.0,80.0
136,202505,76.0,77.0,83.0,82.0,77.0


In [ ]:
df_merged.isnull().sum()

,0
index,0
업황실적BSI,0
매출실적BSI,0
채산성실적BSI,0
자금사정실적BSI,0
인력사정실적BSI,0


In [ ]:
import pandas as pd

# index가 숫자라면 문자열로 변환
df_merged['index'] = df_merged['index'].astype(str)

# 연도와 월 분리
df_merged['연도'] = df_merged['index'].str[:4].astype(int)
df_merged['월'] = df_merged['index'].str[4:6].astype(int)

# 월 → 분기 변환 함수
def month_to_quarter(month):
    if month in [1, 2, 3]:
        return 'Q1'
    elif month in [4, 5, 6]:
        return 'Q2'
    elif month in [7, 8, 9]:
        return 'Q3'
    else:
        return 'Q4'

# 분기 계산
df_merged['분기'] = df_merged['월'].apply(month_to_quarter)

# 분기 인덱스 생성 (예: 2015Q1)
df_merged['index'] = df_merged['연도'].astype(str) + df_merged['분기']

# 분기별 평균 집계
df_quarter = df_merged.groupby('index').mean(numeric_only=True).reset_index()

# 결과 확인
print(df_quarter.head())

    index    업황실적BSI    매출실적BSI   채산성실적BSI  자금사정실적BSI  인력사정실적BSI      연도     월
0  2014Q1  72.333333  85.333333  85.333333  88.000000  89.333333  2014.0   2.0
1  2014Q2  78.000000  92.666667  91.000000  89.666667  89.666667  2014.0   5.0
2  2014Q3  74.666667  88.666667  86.666667  86.333333  90.000000  2014.0   8.0
3  2014Q4  80.000000  94.666667  94.000000  90.333333  94.333333  2014.0  11.0
4  2015Q1  84.333333  92.000000  95.000000  93.000000  89.333333  2015.0   2.0


In [ ]:
df_quarter = df_quarter.drop(columns=['연도', '월'], errors='ignore')

In [ ]:
dfs = [ppi, gdp]
df_final = df_quarter.copy()
key_col = 'index'
for df in dfs:
    df_final = df_final.merge(df, on=key_col, how='left')

df_final

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,gdp
0,2014Q1,72.333333,85.333333,85.333333,88.000000,89.333333,94.67,16590.6
1,2014Q2,78.000000,92.666667,91.000000,89.666667,89.666667,95.01,16624.8
2,2014Q3,74.666667,88.666667,86.666667,86.333333,90.000000,95.08,16808.5
3,2014Q4,80.000000,94.666667,94.000000,90.333333,94.333333,94.60,17112.4
4,2015Q1,84.333333,92.000000,95.000000,93.000000,89.333333,93.75,16893.8
5,2015Q2,80.666667,85.000000,90.666667,95.666667,81.000000,94.21,17061.1
6,2015Q3,73.666667,80.333333,94.333333,96.000000,87.000000,94.64,17504.2
7,2015Q4,74.666667,81.333333,91.333333,89.000000,79.333333,94.47,17520.5
8,2016Q1,64.000000,73.666667,84.000000,85.333333,90.666667,94.59,17465.3
9,2016Q2,71.666667,81.000000,89.333333,91.000000,89.333333,94.43,17655.0


In [ ]:
df_final.to_csv("운수업1차.csv")

## 생산지수

In [ ]:
생산지수 = getKOSIS('DT_1KC2020','Q','201401','202502','T3','101','H')

def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

생산지수['index']=생산지수['PRD_DE'].apply(to_quarter)
생산지수 = 생산지수.drop(columns=["PRD_DE"])
생산지수 = 생산지수.rename(columns={'DT': '생산지수'})
생산지수

,생산지수,index
0,106.6,2014Q1
1,108.4,2014Q2
2,108.7,2014Q3
3,110.6,2014Q4
4,110.6,2015Q1
5,108.3,2015Q2
6,109.4,2015Q3
7,110.9,2015Q4
8,112.6,2016Q1
9,112.6,2016Q2


https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=MGNkMzY3MTdhNjZlMTNiZDMxYjU4NDA4NTdjZWU0YmQ=&itmId=13103114676T.0001+&objL1=13102114676A.01+&objL2=13102114676B.0001+&objL3=&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=M&startPrdDe=201401&endPrdDe=202506&outputFields=OBJ_NM+NM+ITM_NM+PRD_DE+&orgId=146&tblId=DT_MLTM_1314

##내항화물입항현황

In [ ]:
내항화물입항 = getKOSIS('DT_MLTM_1314','M','201401','202506','13103114676T.0001','146','13102114676A.01','13102114676B.0001')

In [ ]:
내항화물입항

,PRD_DE,DT
0,201401,9875715.0
1,201402,8332212.0
2,201403,10323847.0
3,201404,10114454.0
4,201405,9962990.0
...,...,...
133,202502,8043101.0
134,202503,9433027.0
135,202504,9525680.0
136,202505,9472326.0


In [ ]:
import pandas as pd

# index가 숫자라면 문자열로 변환
내항화물입항['PRD_DE'] = 내항화물입항['PRD_DE'].astype(str)

# 연도와 월 분리
내항화물입항['연도'] = 내항화물입항['PRD_DE'].str[:4].astype(int)
내항화물입항['월'] = 내항화물입항['PRD_DE'].str[4:6].astype(int)

# 월 → 분기 변환 함수
def month_to_quarter(month):
    if month in [1, 2, 3]:
        return 'Q1'
    elif month in [4, 5, 6]:
        return 'Q2'
    elif month in [7, 8, 9]:
        return 'Q3'
    else:
        return 'Q4'

# 분기 계산
내항화물입항['분기'] = 내항화물입항['월'].apply(month_to_quarter)

# 분기 인덱스 생성 (예: 2015Q1)
내항화물입항['index'] = 내항화물입항['연도'].astype(str) + 내항화물입항['분기']

# 분기별 평균 집계
내항화물입항 = 내항화물입항.groupby('index').mean(numeric_only=True).reset_index()

# 결과 확인
print(내항화물입항.head())

    index            DT      연도     월
0  2014Q1  9.510591e+06  2014.0   2.0
1  2014Q2  1.009200e+07  2014.0   5.0
2  2014Q3  9.047721e+06  2014.0   8.0
3  2014Q4  9.893506e+06  2014.0  11.0
4  2015Q1  9.264669e+06  2015.0   2.0


In [ ]:
내항화물입항 = 내항화물입항.drop(columns=['연도', '월'], errors='ignore')

In [ ]:
내항화물입항

,index,DT
0,2014Q1,9.510591e+06
1,2014Q2,1.009200e+07
2,2014Q3,9.047721e+06
3,2014Q4,9.893506e+06
4,2015Q1,9.264669e+06
5,2015Q2,1.040028e+07
6,2015Q3,1.023243e+07
7,2015Q4,1.123530e+07
8,2016Q1,1.178885e+07
9,2016Q2,1.217565e+07


https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=MGNkMzY3MTdhNjZlMTNiZDMxYjU4NDA4NTdjZWU0YmQ=&itmId=T001+T002+T003+&objL1=A01+&objL2=B01+&objL3=&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=M&startPrdDe=201401&endPrdDe=202506&outputFields=OBJ_NM+NM+ITM_NM+PRD_DE+&orgId=381&tblId=DT_920005_B001

##항공통계

In [ ]:
항공통계1 = getKOSIS('DT_920005_B001','M','201401','202506','T001','381','A01','B01')

In [ ]:
항공통계2 = getKOSIS('DT_920005_B001','M','201401','202506','T002','381','A01','B01')

In [ ]:
항공통계3 =getKOSIS('DT_920005_B001','M','201401','202506','T003','381','A01','B01')

In [ ]:
항공통계1

,PRD_DE,DT
0,201401,55349.0
1,201402,50317.0
2,201403,56272.0
3,201404,58322.0
4,201405,60847.0
...,...,...
133,202502,66226.0
134,202503,72946.0
135,202504,72575.0
136,202505,75536.0


In [ ]:
import pandas as pd

# index가 숫자라면 문자열로 변환
항공통계1['PRD_DE'] = 항공통계1['PRD_DE'].astype(str)

# 연도와 월 분리
항공통계1['연도'] = 항공통계1['PRD_DE'].str[:4].astype(int)
항공통계1['월'] = 항공통계1['PRD_DE'].str[4:6].astype(int)

# 월 → 분기 변환 함수
def month_to_quarter(month):
    if month in [1, 2, 3]:
        return 'Q1'
    elif month in [4, 5, 6]:
        return 'Q2'
    elif month in [7, 8, 9]:
        return 'Q3'
    else:
        return 'Q4'

# 분기 계산
항공통계1['분기'] = 항공통계1['월'].apply(month_to_quarter)

# 분기 인덱스 생성 (예: 2015Q1)
항공통계1['index'] = 항공통계1['연도'].astype(str) + 항공통계1['분기']

# 분기별 평균 집계
항공통계1 = 항공통계1.groupby('index').mean(numeric_only=True).reset_index()

# 결과 확인
print(항공통계1.head())

    index            DT      연도     월
0  2014Q1  53979.333333  2014.0   2.0
1  2014Q2  59049.000000  2014.0   5.0
2  2014Q3  62013.000000  2014.0   8.0
3  2014Q4  60521.000000  2014.0  11.0
4  2015Q1  60013.333333  2015.0   2.0


In [ ]:
항공통계1 = 항공통계1.drop(columns=['연도', '월'], errors='ignore')

In [ ]:
항공통계2

,PRD_DE,DT
0,201401,8258063.0
1,201402,7842201.0
2,201403,8062608.0
3,201404,8944622.0
4,201405,8585650.0
...,...,...
133,202502,11524668.0
134,202503,12016768.0
135,202504,12327011.0
136,202505,12947261.0


In [ ]:
import pandas as pd

# index가 숫자라면 문자열로 변환
항공통계2['PRD_DE'] = 항공통계2['PRD_DE'].astype(str)

# 연도와 월 분리
항공통계2['연도'] = 항공통계2['PRD_DE'].str[:4].astype(int)
항공통계2['월'] = 항공통계2['PRD_DE'].str[4:6].astype(int)

# 월 → 분기 변환 함수
def month_to_quarter(month):
    if month in [1, 2, 3]:
        return 'Q1'
    elif month in [4, 5, 6]:
        return 'Q2'
    elif month in [7, 8, 9]:
        return 'Q3'
    else:
        return 'Q4'

# 분기 계산
항공통계2['분기'] = 항공통계2['월'].apply(month_to_quarter)

# 분기 인덱스 생성 (예: 2015Q1)
항공통계2['index'] = 항공통계2['연도'].astype(str) + 항공통계2['분기']

# 분기별 평균 집계
항공통계2 = 항공통계2.groupby('index').mean(numeric_only=True).reset_index()

# 결과 확인
print(항공통계2.head())

    index            DT      연도     월
0  2014Q1  8.054291e+06  2014.0   2.0
1  2014Q2  8.790145e+06  2014.0   5.0
2  2014Q3  9.702368e+06  2014.0   8.0
3  2014Q4  9.166528e+06  2014.0  11.0
4  2015Q1  9.356761e+06  2015.0   2.0


In [ ]:
항공통계2 = 항공통계2.drop(columns=['연도', '월'], errors='ignore')

In [ ]:
항공통계3

,PRD_DE,DT
0,201401,317552.55
1,201402,290885.02
2,201403,346672.66
3,201404,328453.26
4,201405,319535.36
...,...,...
133,202502,347012.00
134,202503,399370.00
135,202504,380132.00
136,202505,387939.00


In [ ]:
import pandas as pd

# index가 숫자라면 문자열로 변환
항공통계3['PRD_DE'] = 항공통계3['PRD_DE'].astype(str)

# 연도와 월 분리
항공통계3['연도'] = 항공통계3['PRD_DE'].str[:4].astype(int)
항공통계3['월'] = 항공통계3['PRD_DE'].str[4:6].astype(int)

# 월 → 분기 변환 함수
def month_to_quarter(month):
    if month in [1, 2, 3]:
        return 'Q1'
    elif month in [4, 5, 6]:
        return 'Q2'
    elif month in [7, 8, 9]:
        return 'Q3'
    else:
        return 'Q4'

# 분기 계산
항공통계3['분기'] = 항공통계3['월'].apply(month_to_quarter)

# 분기 인덱스 생성 (예: 2015Q1)
항공통계3['index'] = 항공통계3['연도'].astype(str) + 항공통계3['분기']

# 분기별 평균 집계
항공통계3 = 항공통계3.groupby('index').mean(numeric_only=True).reset_index()

# 결과 확인
print(항공통계3.head())

    index             DT      연도     월
0  2014Q1  318370.076667  2014.0   2.0
1  2014Q2  321957.110000  2014.0   5.0
2  2014Q3  336090.336667  2014.0   8.0
3  2014Q4  349242.876667  2014.0  11.0
4  2015Q1  340106.660000  2015.0   2.0


In [ ]:
항공통계3 = 항공통계3.drop(columns=['연도', '월'], errors='ignore')

In [ ]:
from functools import reduce

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [항공통계1, 항공통계2, 항공통계3]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
항공통계 = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
항공통계

,index,DT_x,DT_y,DT
0,2014Q1,53979.333333,8.054291e+06,318370.076667
1,2014Q2,59049.000000,8.790145e+06,321957.110000
2,2014Q3,62013.000000,9.702368e+06,336090.336667
3,2014Q4,60521.000000,9.166528e+06,349242.876667
4,2015Q1,60013.333333,9.356761e+06,340106.660000
5,2015Q2,63035.000000,9.729616e+06,333065.133333
6,2015Q3,62354.000000,9.987521e+06,334798.116667
7,2015Q4,65657.666667,1.047478e+07,356807.620000
8,2016Q1,65090.333333,1.047449e+07,341487.093333
9,2016Q2,67453.000000,1.130500e+07,354384.063333


In [ ]:
항공통계['항공통계']=항공통계['DT']+항공통계['DT_x']+항공통계['DT_y']

In [ ]:
항공통계 =항공통계.drop(columns=["DT_x","DT_y","DT"])

In [ ]:
항공통계

,index,항공통계
0,2014Q1,8.426640e+06
1,2014Q2,9.171151e+06
2,2014Q3,1.010047e+07
3,2014Q4,9.576292e+06
4,2015Q1,9.756881e+06
5,2015Q2,1.012572e+07
6,2015Q3,1.038467e+07
7,2015Q4,1.089725e+07
8,2016Q1,1.088106e+07
9,2016Q2,1.172683e+07


##기업데이터

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('임지오기업.csv')

In [ ]:
운수업 = df[df['매핑한 산업'] == "운수업"]

In [ ]:
운수업

,Unnamed: 0,업체코드,종목코드,종목명,업체명,사업자번호,상장일,결산월,산업세세분류,조사일,EBITDA,인건비,"유,무형,리스자산의증가",매출원가,수익(매출액),매핑한 산업
816,4488,N710059,A000650,천일고속,(주)천일고속,604-81-00854,1977-06-23,12.0,H49220.시외버스 운송업,2000-01-01,0.000000e+00,1456000.0,0.000000e+00,7.769000e+09,1.106100e+10,운수업
817,4489,N710059,A000650,천일고속,(주)천일고속,604-81-00854,1977-06-23,12.0,H49220.시외버스 운송업,2000-04-01,0.000000e+00,6936863.0,0.000000e+00,8.123852e+09,1.061285e+10,운수업
818,4490,N710059,A000650,천일고속,(주)천일고속,604-81-00854,1977-06-23,12.0,H49220.시외버스 운송업,2000-07-01,0.000000e+00,-3657863.0,0.000000e+00,8.831148e+09,1.224315e+10,운수업
819,4491,N710059,A000650,천일고속,(주)천일고속,604-81-00854,1977-06-23,12.0,H49220.시외버스 운송업,2000-10-01,3.907068e+09,13533696.0,1.355107e+09,1.118816e+10,1.162803e+10,운수업
820,4492,N710059,A000650,천일고속,(주)천일고속,604-81-00854,1977-06-23,12.0,H49220.시외버스 운송업,2001-01-01,0.000000e+00,1623000.0,0.000000e+00,8.750000e+09,1.134500e+10,운수업
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69865,342307,NQJ8766,A465770,STX그린로지스,(주)STX그린로지스,423-86-02883,2023-09-15,12.0,H50112.외항 화물 운송업,2024-04-01,4.281625e+09,301399.0,7.200000e+04,6.702848e+09,8.582526e+09,운수업
69866,342308,NQJ8766,A465770,STX그린로지스,(주)STX그린로지스,423-86-02883,2023-09-15,12.0,H50112.외항 화물 운송업,2024-07-01,6.431060e+09,487543.0,7.870000e+05,8.147721e+09,1.163277e+10,운수업
69867,342309,NQJ8766,A465770,STX그린로지스,(주)STX그린로지스,423-86-02883,2023-09-15,12.0,H50112.외항 화물 운송업,2024-10-01,6.189030e+09,-1144365.0,5.385085e+09,1.125232e+10,1.304622e+10,운수업
69868,342310,NQJ8766,A465770,STX그린로지스,(주)STX그린로지스,423-86-02883,2023-09-15,12.0,H50112.외항 화물 운송업,2025-01-01,-1.584392e+09,0.0,7.168050e+08,1.937421e+10,1.879805e+10,운수업


In [ ]:
columns = ['EBITDA', '인건비']
grouped = 운수업.groupby('조사일')[columns].sum()
grouped

,EBITDA,인건비
조사일,,
2000-01-01,0.000000e+00,8.079182e+07
2000-04-01,0.000000e+00,6.626789e+08
2000-07-01,0.000000e+00,-4.816447e+08
2000-10-01,1.856816e+12,1.534067e+09
2001-01-01,0.000000e+00,9.455113e+07
...,...,...
2024-04-01,2.704688e+12,3.402533e+08
2024-07-01,4.162786e+12,3.305415e+08
2024-10-01,3.681266e+12,2.488330e+09


In [ ]:
grouped = grouped.reset_index()

In [ ]:
grouped

,조사일,EBITDA,인건비
0,2000-01-01,0.000000e+00,8.079182e+07
1,2000-04-01,0.000000e+00,6.626789e+08
2,2000-07-01,0.000000e+00,-4.816447e+08
3,2000-10-01,1.856816e+12,1.534067e+09
4,2001-01-01,0.000000e+00,9.455113e+07
...,...,...,...
97,2024-04-01,2.704688e+12,3.402533e+08
98,2024-07-01,4.162786e+12,3.305415e+08
99,2024-10-01,3.681266e+12,2.488330e+09
100,2025-01-01,2.847633e+12,8.013731e+07


In [ ]:
grouped['조사일']=pd.to_datetime(grouped['조사일'])

# 분기(Quarter) 단위로 변환
grouped["index"] = grouped["조사일"].dt.to_period("Q")
grouped = grouped.drop(columns=["조사일"])
print(grouped)

           EBITDA           인건비   index
0    0.000000e+00  8.079182e+07  2000Q1
1    0.000000e+00  6.626789e+08  2000Q2
2    0.000000e+00 -4.816447e+08  2000Q3
3    1.856816e+12  1.534067e+09  2000Q4
4    0.000000e+00  9.455113e+07  2001Q1
..            ...           ...     ...
97   2.704688e+12  3.402533e+08  2024Q2
98   4.162786e+12  3.305415e+08  2024Q3
99   3.681266e+12  2.488330e+09  2024Q4
100  2.847633e+12  8.013731e+07  2025Q1
101  2.294565e+12 -8.013731e+07  2025Q2

[102 rows x 3 columns]


In [ ]:
mask = (grouped["index"] >= pd.Period("2014Q1")) & (grouped["index"] <= pd.Period("2025Q2"))
기업= grouped.loc[mask]
기업
기업['합산'] = 기업['EBITDA']+기업['인건비']
기업

/tmp/ipython-input-1389489221.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  기업['합산'] = 기업['EBITDA']+기업['인건비']


,EBITDA,인건비,index,합산
56,7.707092e+11,1.940276e+08,2014Q1,7.709033e+11
57,7.822086e+11,1.777419e+08,2014Q2,7.823864e+11
58,1.103013e+12,1.714626e+08,2014Q3,1.103185e+12
59,1.080872e+12,6.397593e+08,2014Q4,1.081512e+12
60,1.191044e+12,1.947361e+08,2015Q1,1.191239e+12
61,7.540427e+11,1.939857e+08,2015Q2,7.542367e+11
62,1.175376e+12,1.868632e+08,2015Q3,1.175563e+12
63,1.260069e+12,2.255290e+09,2015Q4,1.262324e+12
64,1.121607e+12,1.986590e+08,2016Q1,1.121806e+12
65,8.353342e+11,1.975198e+08,2016Q2,8.355317e+11


In [ ]:
기업 = 기업[['index'] + [col for col in 기업.columns if col != 'index']]
기업 = 기업.reset_index()

In [ ]:
기업

,level_0,index,EBITDA,인건비,합산
0,56,2014Q1,7.707092e+11,1.940276e+08,7.709033e+11
1,57,2014Q2,7.822086e+11,1.777419e+08,7.823864e+11
2,58,2014Q3,1.103013e+12,1.714626e+08,1.103185e+12
3,59,2014Q4,1.080872e+12,6.397593e+08,1.081512e+12
4,60,2015Q1,1.191044e+12,1.947361e+08,1.191239e+12
5,61,2015Q2,7.540427e+11,1.939857e+08,7.542367e+11
6,62,2015Q3,1.175376e+12,1.868632e+08,1.175563e+12
7,63,2015Q4,1.260069e+12,2.255290e+09,1.262324e+12
8,64,2016Q1,1.121607e+12,1.986590e+08,1.121806e+12
9,65,2016Q2,8.353342e+11,1.975198e+08,8.355317e+11


In [ ]:
기업 = 기업.drop(columns=['level_0'])

In [ ]:
기업

,index,EBITDA,인건비,합산
0,2014Q1,7.707092e+11,1.940276e+08,7.709033e+11
1,2014Q2,7.822086e+11,1.777419e+08,7.823864e+11
2,2014Q3,1.103013e+12,1.714626e+08,1.103185e+12
3,2014Q4,1.080872e+12,6.397593e+08,1.081512e+12
4,2015Q1,1.191044e+12,1.947361e+08,1.191239e+12
5,2015Q2,7.540427e+11,1.939857e+08,7.542367e+11
6,2015Q3,1.175376e+12,1.868632e+08,1.175563e+12
7,2015Q4,1.260069e+12,2.255290e+09,1.262324e+12
8,2016Q1,1.121607e+12,1.986590e+08,1.121806e+12
9,2016Q2,8.353342e+11,1.975198e+08,8.355317e+11


##전체 데이터 합치기

In [ ]:
from functools import reduce
import pandas as pd

# 모든 데이터프레임의 index 컬럼을 문자열로 변환
for df in [df_final,생산지수, 내항화물입항, 항공통계, 기업]:
    df['index'] = df['index'].astype(str)

# 리스트에 넣고 합치기
dfs = [df_final,생산지수, 내항화물입항, 항공통계, 기업]
df_merged = reduce(lambda left, right: pd.merge(left, right, on='index', how='outer'), dfs)


In [ ]:
df_merged
df_merged = df_merged[['index'] + [col for col in df_merged.columns if col != 'index']]

In [ ]:
df_merged
df_merged = df_merged[['index'] + [col for col in df_merged.columns if col != 'index']]

In [ ]:
df_merged
df_merged = df_merged.rename(columns={'DT': '내항화물입항'})

In [ ]:
df_merged

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,gdp,생산지수,내항화물입항,항공통계,EBITDA,인건비,합산
0,2014Q1,72.333333,85.333333,85.333333,88.000000,89.333333,94.67,16590.6,106.6,9.510591e+06,8.426640e+06,7.707092e+11,1.940276e+08,7.709033e+11
1,2014Q2,78.000000,92.666667,91.000000,89.666667,89.666667,95.01,16624.8,108.4,1.009200e+07,9.171151e+06,7.822086e+11,1.777419e+08,7.823864e+11
2,2014Q3,74.666667,88.666667,86.666667,86.333333,90.000000,95.08,16808.5,108.7,9.047721e+06,1.010047e+07,1.103013e+12,1.714626e+08,1.103185e+12
3,2014Q4,80.000000,94.666667,94.000000,90.333333,94.333333,94.60,17112.4,110.6,9.893506e+06,9.576292e+06,1.080872e+12,6.397593e+08,1.081512e+12
4,2015Q1,84.333333,92.000000,95.000000,93.000000,89.333333,93.75,16893.8,110.6,9.264669e+06,9.756881e+06,1.191044e+12,1.947361e+08,1.191239e+12
5,2015Q2,80.666667,85.000000,90.666667,95.666667,81.000000,94.21,17061.1,108.3,1.040028e+07,1.012572e+07,7.540427e+11,1.939857e+08,7.542367e+11
6,2015Q3,73.666667,80.333333,94.333333,96.000000,87.000000,94.64,17504.2,109.4,1.023243e+07,1.038467e+07,1.175376e+12,1.868632e+08,1.175563e+12
7,2015Q4,74.666667,81.333333,91.333333,89.000000,79.333333,94.47,17520.5,110.9,1.123530e+07,1.089725e+07,1.260069e+12,2.255290e+09,1.262324e+12
8,2016Q1,64.000000,73.666667,84.000000,85.333333,90.666667,94.59,17465.3,112.6,1.178885e+07,1.088106e+07,1.121607e+12,1.986590e+08,1.121806e+12
9,2016Q2,71.666667,81.000000,89.333333,91.000000,89.333333,94.43,17655.0,112.6,1.217565e+07,1.172683e+07,8.353342e+11,1.975198e+08,8.355317e+11


In [ ]:
df_merged.to_csv('운수업.csv')

#데이터qoq변환

In [ ]:
import pandas as pd

# df는 위의 데이터프레임
df_qoq = df_merged.copy()

# 분기 순서대로 정렬 (혹시 index가 문자열일 경우를 대비)
df_qoq = df_qoq.sort_values('index')

# 수치형 컬럼만 선택 (index 제외)
num_cols = df_qoq.select_dtypes(include=['number']).columns

# 전분기 대비 증감률 (%) 계산
df_qoq[num_cols] = df_qoq[num_cols].pct_change() * 100

# 결과 확인
print(df_qoq.head())

    index   업황실적BSI   매출실적BSI  채산성실적BSI  자금사정실적BSI  인력사정실적BSI       ppi  \
0  2014Q1       NaN       NaN       NaN        NaN        NaN       NaN   
1  2014Q2  7.834101  8.593750  6.640625   1.893939   0.373134  0.359142   
2  2014Q3 -4.273504 -4.316547 -4.761905  -3.717472   0.371747  0.073676   
3  2014Q4  7.142857  6.766917  8.461538   4.633205   4.814815 -0.504838   
4  2015Q1  5.416667 -2.816901  1.063830   2.952030  -5.300353 -0.898520   

        gdp      생산지수     내항화물입항       항공통계     EBITDA         인건비         합산  
0       NaN       NaN        NaN        NaN        NaN         NaN        NaN  
1  0.206141  1.688555   6.113307   8.835202   1.492054   -8.393527   1.489566  
2  1.104976  0.276753 -10.347615  10.133082  41.012682   -3.532787  41.002562  
3  1.808014  1.747930   9.348037  -5.189650  -2.007338  273.118785  -1.964576  
4 -1.277436  0.000000  -6.356058   1.885794  10.192892  -69.561028  10.145714  


In [ ]:
df_qoq
df_qoq = df_qoq.drop(index=[0, 1, 2,3])

In [ ]:
df_qoq.isnull().sum()

,0
index,0
업황실적BSI,0
매출실적BSI,0
채산성실적BSI,0
자금사정실적BSI,0
인력사정실적BSI,0
ppi,0
gdp,0
생산지수,0
내항화물입항,0


In [ ]:
df_qoq.to_csv('운수업qoq.csv')

#데이터yoy변환

In [ ]:
import pandas as pd

# df_merged는 원본 데이터프레임
df_yoy = df_merged.copy()

# 분기 순서대로 정렬
df_yoy = df_yoy.sort_values('index')

# 수치형 컬럼만 선택 (index 제외)
num_cols = df_yoy.select_dtypes(include=['number']).columns

# 전년 동분기 대비 증감률 (%) 계산 (4분기 차이)
df_yoy[num_cols] = df_yoy[num_cols].pct_change(periods=4) * 100

# 결과 확인
print(df_yoy.head(10))

    index    업황실적BSI    매출실적BSI   채산성실적BSI  자금사정실적BSI  인력사정실적BSI       ppi  \
0  2014Q1        NaN        NaN        NaN        NaN        NaN       NaN   
1  2014Q2        NaN        NaN        NaN        NaN        NaN       NaN   
2  2014Q3        NaN        NaN        NaN        NaN        NaN       NaN   
3  2014Q4        NaN        NaN        NaN        NaN        NaN       NaN   
4  2015Q1  16.589862   7.812500  11.328125   5.681818   0.000000 -0.971797   
5  2015Q2   3.418803  -8.273381  -0.366300   6.691450  -9.665428 -0.842017   
6  2015Q3  -1.339286  -9.398496   8.846154  11.196911  -3.333333 -0.462768   
7  2015Q4  -6.666667 -14.084507  -2.836879  -1.476015 -15.901060 -0.137421   
8  2016Q1 -24.110672 -19.927536 -11.578947  -8.243728   1.492537  0.896000   
9  2016Q2 -11.157025  -4.705882  -1.470588  -4.878049  10.288066  0.233521   

        gdp      생산지수     내항화물입항       항공통계     EBITDA         인건비         합산  
0       NaN       NaN        NaN        NaN        NaN       

In [ ]:
df_yoy=df_yoy.dropna()

In [ ]:
df_yoy

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,gdp,생산지수,내항화물입항,항공통계,EBITDA,인건비,합산
4,2015Q1,16.589862,7.812500,11.328125,5.681818,0.000000,-0.971797,1.827541,3.752345,-2.585777,15.786137,54.538734,0.365162,54.525099
5,2015Q2,3.418803,-8.273381,-0.366300,6.691450,-9.665428,-0.842017,2.624392,-0.092251,3.054669,10.408345,-3.600821,9.139019,-3.597927
6,2015Q3,-1.339286,-9.398496,8.846154,11.196911,-3.333333,-0.462768,4.138977,0.643974,13.093975,2.813748,6.560469,8.981904,6.560846
7,2015Q4,-6.666667,-14.084507,-2.836879,-1.476015,-15.901060,-0.137421,2.384820,0.271248,13.562388,13.794028,16.578906,252.521719,16.718476
8,2016Q1,-24.110672,-19.927536,-11.578947,-8.243728,1.492537,0.896000,3.382898,1.808318,27.245270,11.521952,-5.829961,2.014453,-5.828678
9,2016Q2,-11.157025,-4.705882,-1.470588,-4.878049,10.288066,0.233521,3.481018,3.970452,17.070424,15.812389,10.780759,1.821794,10.778455
10,2016Q3,-4.072398,-4.149378,-9.540636,-12.500000,3.448276,-0.052832,0.221661,3.199269,12.223408,22.023830,8.877472,1.942869,8.876370
11,2016Q4,-8.482143,2.459016,-6.204380,-4.868914,12.605042,0.423415,-1.978825,-0.541028,9.429778,8.998869,-15.017777,4.236630,-14.983376
12,2017Q1,11.458333,18.552036,2.777778,0.390625,-6.250000,0.306586,2.230709,0.444050,17.307130,9.558659,-20.255171,-1.010147,-20.251763
13,2017Q2,9.302326,10.699588,1.865672,0.366300,-8.582090,1.111935,0.971396,0.799290,-0.267934,4.202472,16.629149,4.695818,16.626328


In [ ]:
df_yoy.to_csv("운수업yoy.csv")

#상관계수보기(qoq)

In [ ]:
qoq = pd.read_csv('운수업qoq.csv')
qoq = qoq.drop(columns=["Unnamed: 0"])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# GDP_QoQ 기준
target = 'gdp'

In [ ]:
# 수치형 컬럼만 선택
num_cols = qoq.select_dtypes(include=['number']).columns

# GDP 기준 상관계수 계산
corr = qoq[num_cols].corr()[target].sort_values(ascending=False)

print("📊 GDP 대비 단순 상관계수")
print(corr)


📊 GDP 대비 단순 상관계수
gdp          1.000000
생산지수         0.764214
항공통계         0.673314
채산성실적BSI     0.571004
자금사정실적BSI    0.552945
업황실적BSI      0.531511
매출실적BSI      0.515545
내항화물입항       0.228764
ppi          0.142010
인건비          0.068334
합산          -0.049867
EBITDA      -0.050058
인력사정실적BSI   -0.204018
Name: gdp, dtype: float64


In [ ]:
# df2 사본
df2 = qoq.copy()

# target 및 feature 지정
target_col = 'gdp'
feature_cols = [col for col in df2.columns if col not in ['index', target_col]]

# 결과 저장용 데이터프레임
corr_df = pd.DataFrame(columns=['Feature', 'Lag', 'Correlation'])

# lag 1~4 생성 및 상관계수 계산
for col in feature_cols:
    for lag in range(1, 5):  # lag 1~4
        lag_col_name = f"{col}_lag{lag}"
        df2[lag_col_name] = df2[col].shift(lag)  # 실제 데이터에 lag 컬럼 추가
        corr = df2[target_col].corr(df2[lag_col_name])  # 상관계수 계산
        corr_df = pd.concat(
            [corr_df, pd.DataFrame({'Feature': [col], 'Lag': [lag], 'Correlation': [corr]})],
            ignore_index=True
        )

# 상관계수 높은 순으로 정렬
corr_df.sort_values(by='Correlation', ascending=False, inplace=True)
corr_df.reset_index(drop=True, inplace=True)

corr_df

/tmp/ipython-input-1052179002.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  corr_df = pd.concat(


,Feature,Lag,Correlation
0,자금사정실적BSI,1,0.243163
1,ppi,3,0.230088
2,업황실적BSI,1,0.229794
3,인력사정실적BSI,4,0.214989
4,업황실적BSI,2,0.210957
5,합산,4,0.207337
6,EBITDA,4,0.207216
7,생산지수,2,0.201907
8,항공통계,2,0.190231
9,ppi,4,0.187018


In [ ]:
df2

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,gdp,생산지수,내항화물입항,...,EBITDA_lag3,EBITDA_lag4,인건비_lag1,인건비_lag2,인건비_lag3,인건비_lag4,합산_lag1,합산_lag2,합산_lag3,합산_lag4
0,2015Q1,5.416667,-2.816901,1.063830,2.952030,-5.300353,-0.898520,-1.277436,0.000000,-6.356058,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015Q2,-4.347826,-7.608696,-4.561404,2.867384,-9.328358,0.490667,0.990304,-2.079566,12.257445,...,NaN,NaN,-69.561028,NaN,NaN,NaN,10.145714,NaN,NaN,NaN
2,2015Q3,-8.677686,-5.490196,4.044118,0.348432,7.407407,0.456427,2.597136,1.015697,-1.613924,...,NaN,NaN,-0.385349,-69.561028,NaN,NaN,-36.684690,10.145714,NaN,NaN
3,2015Q4,1.357466,1.244813,-3.180212,-7.291667,-8.812261,-0.179628,0.093121,1.371115,9.800936,...,10.192892,NaN,-3.671660,-0.385349,-69.561028,NaN,55.861297,-36.684690,10.145714,NaN
4,2016Q1,-14.285714,-9.426230,-8.029197,-4.119850,14.285714,0.127024,-0.315060,1.532913,4.926894,...,-36.690625,10.192892,1106.920327,-3.671660,-0.385349,-69.561028,7.380390,55.861297,-36.684690,10.145714
5,2016Q2,11.979167,9.954751,6.349206,6.640625,-1.470588,-0.169151,1.086154,0.000000,3.281063,...,55.876613,-36.690625,-91.191422,1106.920327,-3.671660,-0.385349,-11.131743,7.380390,55.861297,-36.684690
6,2016Q3,-1.395349,-4.938272,-4.477612,-7.692308,0.746269,0.169438,-0.634381,0.266430,-5.687361,...,7.205584,55.876613,-0.573476,-91.191422,1106.920327,-3.671660,-25.519021,-11.131743,7.380390,55.861297
7,2016Q4,-3.301887,8.225108,0.390625,0.793651,-0.740741,0.296014,-2.104543,-2.302923,7.067610,...,-10.988451,7.205584,-3.557117,-0.573476,-91.191422,1106.920327,53.185132,-25.519021,-11.131743,7.380390
8,2017Q1,4.390244,4.800000,0.778210,1.181102,-4.850746,0.010541,3.965925,2.538531,12.480103,...,-25.523439,-10.988451,1134.076579,-3.557117,-0.573476,-91.191422,-16.151519,53.185132,-25.519021,-11.131743
9,2017Q2,9.813084,2.671756,5.405405,6.614786,-3.921569,0.632378,-0.159060,0.353669,-12.192602,...,53.198549,-25.523439,-91.634804,1134.076579,-3.557117,-0.573476,-16.638811,-16.151519,53.185132,-25.519021


In [ ]:
# 숫자형 컬럼만 선택 (연도분기 제외)
numeric_df = df2.select_dtypes(include=["number"])

# 전체 상관계수
corr = numeric_df.corr()

target_corr = corr["gdp"].drop("gdp")
target_corr = target_corr.sort_values(ascending=False)
target_corr

,gdp
생산지수,0.764214
항공통계,0.673314
채산성실적BSI,0.571004
자금사정실적BSI,0.552945
업황실적BSI,0.531511
매출실적BSI,0.515545
자금사정실적BSI_lag1,0.243163
ppi_lag3,0.230088
업황실적BSI_lag1,0.229794
내항화물입항,0.228764


In [ ]:
target_corr.to_csv('운수업순서.csv')

In [ ]:
df2.dropna(inplace=True)

In [ ]:
df2

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,gdp,생산지수,내항화물입항,...,EBITDA_lag3,EBITDA_lag4,인건비_lag1,인건비_lag2,인건비_lag3,인건비_lag4,합산_lag1,합산_lag2,합산_lag3,합산_lag4
4,2016Q1,-14.285714,-9.426230,-8.029197,-4.119850,14.285714,0.127024,-0.315060,1.532913,4.926894,...,-36.690625,10.192892,1106.920327,-3.671660,-0.385349,-69.561028,7.380390,55.861297,-36.684690,10.145714
5,2016Q2,11.979167,9.954751,6.349206,6.640625,-1.470588,-0.169151,1.086154,0.000000,3.281063,...,55.876613,-36.690625,-91.191422,1106.920327,-3.671660,-0.385349,-11.131743,7.380390,55.861297,-36.684690
6,2016Q3,-1.395349,-4.938272,-4.477612,-7.692308,0.746269,0.169438,-0.634381,0.266430,-5.687361,...,7.205584,55.876613,-0.573476,-91.191422,1106.920327,-3.671660,-25.519021,-11.131743,7.380390,55.861297
7,2016Q4,-3.301887,8.225108,0.390625,0.793651,-0.740741,0.296014,-2.104543,-2.302923,7.067610,...,-10.988451,7.205584,-3.557117,-0.573476,-91.191422,1106.920327,53.185132,-25.519021,-11.131743,7.380390
8,2017Q1,4.390244,4.800000,0.778210,1.181102,-4.850746,0.010541,3.965925,2.538531,12.480103,...,-25.523439,-10.988451,1134.076579,-3.557117,-0.573476,-91.191422,-16.151519,53.185132,-25.519021,-11.131743
9,2017Q2,9.813084,2.671756,5.405405,6.614786,-3.921569,0.632378,-0.159060,0.353669,-12.192602,...,53.198549,-25.523439,-91.634804,1134.076579,-3.557117,-0.573476,-16.638811,-16.151519,53.185132,-25.519021
10,2017Q3,0.000000,2.230483,0.732601,-1.824818,4.489796,0.387516,1.312653,1.409692,-10.317659,...,-16.322737,53.198549,5.157660,-91.634804,1134.076579,-3.557117,8.923324,-16.638811,-16.151519,53.185132
11,2017Q4,0.000000,2.909091,-6.545455,-3.345725,1.171875,0.427752,2.012126,-0.608167,-2.051382,...,-16.474170,-16.322737,-8.968019,5.157660,-91.634804,1134.076579,28.807492,8.923324,-16.638811,-16.151519
12,2018Q1,0.851064,0.000000,1.167315,-2.692308,0.772201,0.425930,-0.160661,0.087413,-6.434059,...,8.924152,-16.474170,744.565922,-8.968019,5.157660,-91.634804,24.781158,28.807492,8.923324,-16.638811
13,2018Q2,5.063291,0.353357,7.692308,8.300395,-0.383142,-0.289645,2.594282,3.144105,7.915796,...,28.815511,8.924152,-86.078690,744.565922,-8.968019,5.157660,-38.918222,24.781158,28.807492,8.923324


In [ ]:
df2 = df2.reset_index()
df2 = df2.drop(columns=["level_0"])

In [ ]:
df2.to_csv('운수업lag.csv', encoding='utf-8-sig')

#ARIMA

'생산지수','항공통계','채산성실적BSI','자금사정실적BSI','업황실적BSI','매출실적BSI','EBITDA_lag2','합산_lag2'

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('운수업lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['생산지수','항공통계','채산성실적BSI','자금사정실적BSI','업황실적BSI','매출실적BSI','EBITDA_lag2','합산_lag2']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (0.0, 1.0, 2.0)
📉 평균 Train MAE: 1.9586
📈 평균 Test MAE: 4.3742


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('운수업lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['생산지수','항공통계','채산성실적BSI','자금사정실적BSI','업황실적BSI','매출실적BSI','EBITDA_lag2']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (1.0, 1.0, 0.0)
📉 평균 Train MAE: 2.0045
📈 평균 Test MAE: 3.8307


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('운수업lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['생산지수','항공통계','채산성실적BSI','자금사정실적BSI','업황실적BSI','매출실적BSI']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (0.0, 1.0, 1.0)
📉 평균 Train MAE: 2.2972
📈 평균 Test MAE: 3.6507


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('운수업lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['생산지수','항공통계','채산성실적BSI','자금사정실적BSI','업황실적BSI']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (0.0, 1.0, 1.0)
📉 평균 Train MAE: 2.3274
📈 평균 Test MAE: 2.841


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('운수업lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['생산지수','항공통계','채산성실적BSI','자금사정실적BSI']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (0.0, 1.0, 1.0)
📉 평균 Train MAE: 2.2186
📈 평균 Test MAE: 2.7004


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('운수업lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['생산지수','항공통계','채산성실적BSI']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (0.0, 1.0, 1.0)
📉 평균 Train MAE: 2.1855
📈 평균 Test MAE: 2.272


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('운수업lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['생산지수','항공통계']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (0.0, 1.0, 1.0)
📉 평균 Train MAE: 2.1081
📈 평균 Test MAE: 2.2712


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('운수업lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['생산지수']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (0.0, 1.0, 1.0)
📉 평균 Train MAE: 2.2084
📈 평균 Test MAE: 2.1378


## 아리마 예측값

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('운수업lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','항공통계']

y = df[target].dropna()
X = df[exog_vars].dropna()

df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    pred = fit.forecast(steps=1, exog=X_test)

                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 최적 (p,d,q) 선택
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

best_p, best_d, best_q = int(best['p']), int(best['d']), int(best['q'])

print("✅ 최적 ARIMAX(p,d,q):", (best_p, best_d, best_q))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

# ---------------------------------------------------
# 8️⃣ 최적 (p,d,q)로 전체 구간 롤링 예측값 저장
# ---------------------------------------------------
rolling_preds = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    train_data = y.loc[train_start:train_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(train_data) < train_window or len(X_test) == 0:
        continue

    try:
        model = SARIMAX(train_data, exog=X_train, order=(best_p, best_d, best_q),
                        enforce_stationarity=False, enforce_invertibility=False)
        fit = model.fit(disp=False)
        pred = fit.forecast(steps=1, exog=X_test)

        rolling_preds.append({
            'date': test_end,
            'pred': float(pred),
            'actual': float(y.loc[test_end])
        })
    except:
        continue

pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 최적 모델 예측값(pred) 시계열:")
print(pred_df)


✅ 최적 ARIMAX(p,d,q): (0, 1, 1)
📉 평균 Train MAE: 2.1081
📈 평균 Test MAE: 2.2712

📌 최적 모델 예측값(pred) 시계열:
            pred    actual
date                      
2021Q1 -1.603261  7.685563
2021Q2  4.995760  2.869762
2021Q3  0.557180 -0.316051
2021Q4  5.444280  2.610384
2022Q1  0.228561  2.658985
2022Q2  6.982936  5.777034
2022Q3  3.370064  2.496486
2022Q4  2.523604  4.233273
2023Q1  0.255713  0.323446
2023Q2  3.217894  9.975364
2023Q3  4.228551  2.870310
2023Q4  1.683536  3.453564
2024Q1  0.878941 -1.827102
2024Q2  2.052545  3.401592
2024Q3  2.343575  2.824389
2024Q4  2.213575  0.650825
2025Q1  0.422373 -2.883479
2025Q2  2.244876  2.063035


In [ ]:
level_2025Q1 = 26223.5

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 26812.18510415539


#랜덤포레스트

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운수업lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','항공통계']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [8, 10, 12],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

# 첫 번째 윈도우 구간 설정
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

# 하이퍼파라미터 탐색 (단 1회)
rf = RandomForestRegressor(random_state=42)
rand_search = RandomizedSearchCV(
    rf,
    param_distributions=param_grid,
    n_iter=10,            # 108조합 중 10개만 탐색
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 고정된 최적 파라미터로 학습
        model = RandomForestRegressor(**best_params, random_state=42)
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE 계산
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 랜덤포레스트 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'n_estimators': 400, 'min_samples_split': 5, 'max_features': 'sqrt', 'max_depth': 12}

📊 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 1.1089
📈 평균 Test MAE: 1.9752


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운수업lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [8, 10, 12],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

# 첫 번째 윈도우 구간 설정
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

# 하이퍼파라미터 탐색 (단 1회)
rf = RandomForestRegressor(random_state=42)
rand_search = RandomizedSearchCV(
    rf,
    param_distributions=param_grid,
    n_iter=10,            # 108조합 중 10개만 탐색
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 고정된 최적 파라미터로 학습
        model = RandomForestRegressor(**best_params, random_state=42)
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE 계산
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 랜덤포레스트 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'n_estimators': 400, 'min_samples_split': 5, 'max_features': 'sqrt', 'max_depth': 12}

📊 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 1.2734
📈 평균 Test MAE: 2.0551


## 랜포 예측값

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운수업lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ y, X 설정
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수', '항공통계'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ RandomizedSearchCV (초기 구간 1회)
# -----------------------------
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [8, 10, 12],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

rf = RandomForestRegressor(random_state=42)
rand_search = RandomizedSearchCV(
    rf,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 + pred 저장
# -----------------------------
train_mae_list = []
test_mae_list = []
rolling_preds = []   # ⭐ 예측값 저장 리스트

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = RandomForestRegressor(**best_params, random_state=42)
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(test_pred),
            'actual': float(y_test.values[0])
        })

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 정리
# -----------------------------
print("\n📊 랜덤포레스트 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 랜덤포레스트 예측값(pred) 시계열:")
print(pred_df)


✅ 최적 하이퍼파라미터: {'n_estimators': 400, 'min_samples_split': 5, 'max_features': 'sqrt', 'max_depth': 12}

📊 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 1.1089
📈 평균 Test MAE: 1.9752

📌 랜덤포레스트 예측값(pred) 시계열:
            pred    actual
date                      
2021Q1  0.003890  7.685563
2021Q2  3.322058  2.869762
2021Q3  3.125054 -0.316051
2021Q4  3.126827  2.610384
2022Q1  3.046933  2.658985
2022Q2  3.014061  5.777034
2022Q3  2.906093  2.496486
2022Q4  2.597474  4.233273
2023Q1  1.400381  0.323446
2023Q2  3.068972  9.975364
2023Q3  3.415155  2.870310
2023Q4  1.393535  3.453564
2024Q1  0.169812 -1.827102
2024Q2  2.243924  3.401592
2024Q3  2.687222  2.824389
2024Q4  2.567573  0.650825
2025Q1 -0.708841 -2.883479
2025Q2  2.357777  2.063035


In [ ]:
level_2025Q1 = 26223.5

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 26841.791655780762


#AR

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운수업qoq.csv')

# GDP 단일 시계열만 사용 (AR(1))
y = df['gdp'].dropna()
y.index = pd.PeriodIndex(df.loc[y.index, 'index'], freq='Q')

# -----------------------------
# 2️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

train_mae_list = []
test_mae_list = []

# -----------------------------
# 3️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    train_data = y.loc[train_start:train_end]
    test_data = y.loc[test_end:test_end]

    if len(train_data) < train_window:
        continue

    try:
        # AR(1) 학습
        model = AutoReg(train_data, lags=1, old_names=False)
        fit = model.fit()

        # --- Train 예측 ---
        # fittedvalues는 길이가 n-1이므로 첫 번째 값 제외하고 MAE 계산
        train_mae = mean_absolute_error(train_data[1:], fit.fittedvalues)
        train_mae_list.append(train_mae)

        # --- Test 예측 (1분기 앞 예측) ---
        pred = fit.predict(start=len(train_data), end=len(train_data))
        test_mae = mean_absolute_error(test_data, pred)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 4️⃣ 결과 출력
# -----------------------------
print("✅ AR(1) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ AR(1) 롤링 예측 결과
📉 평균 Train MAE: 2.7858
📈 평균 Test MAE: 4.1790


## AR 예측값

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운수업qoq.csv')

# GDP 단일 시계열
y = df['gdp'].dropna()
y.index = pd.PeriodIndex(df.loc[y.index, 'index'], freq='Q')

# -----------------------------
# 2️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

train_mae_list = []
test_mae_list = []

# ⭐ 예측값 저장할 리스트
rolling_preds = []

# -----------------------------
# 3️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    train_data = y.loc[train_start:train_end]
    test_data = y.loc[test_end:test_end]

    if len(train_data) < train_window:
        continue

    try:
        # AR(1) 학습
        model = AutoReg(train_data, lags=1, old_names=False)
        fit = model.fit()

        # --- Train MAE ---
        train_mae = mean_absolute_error(train_data[1:], fit.fittedvalues)
        train_mae_list.append(train_mae)

        # --- Test 예측 ---
        pred = fit.predict(start=len(train_data), end=len(train_data))
        test_mae = mean_absolute_error(test_data, pred)
        test_mae_list.append(test_mae)

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(pred),
            'actual': float(test_data.values[0])
        })

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 4️⃣ 결과 출력
# -----------------------------
print("✅ AR(1) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

# -----------------------------
# 5️⃣ pred 시계열 데이터프레임 출력
# -----------------------------
pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 AR(1) 예측값(pred) 시계열:")
print(pred_df)


✅ AR(1) 롤링 예측 결과
📉 평균 Train MAE: 2.7858
📈 평균 Test MAE: 4.1790

📌 AR(1) 예측값(pred) 시계열:
            pred     actual
date                       
2020Q1  0.292052  -9.333361
2020Q2  6.845167 -11.299269
2020Q3 -7.214369   4.722017
2020Q4  0.222201   1.057180
2021Q1 -0.036291   7.685563
2021Q2  1.263077   2.869762
2021Q3  0.834852  -0.316051
2021Q4  0.344671   2.610384
2022Q1  0.782617   2.658985
2022Q2  1.010792   5.777034
2022Q3  2.038691   2.496486
2022Q4  1.219299   4.233273
2023Q1  1.936661   0.323446
2023Q2  0.722883   9.975364
2023Q3  3.666520   2.870310
2023Q4  1.778796   3.453564
2024Q1  2.178607  -1.827102
2024Q2  0.813306   3.401592
2024Q3  2.120977   2.824389
2024Q4  2.017973   0.650825
2025Q1  2.085432  -2.883479
2025Q2  3.629592   2.063035


In [ ]:
level_2025Q1 = 26223.5

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 27175.306064964127


#XGB

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운수업lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','항공통계'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 하이퍼파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3]
}

# 첫 번째 윈도우로 탐색
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

# RandomizedSearchCV 실행
xgb = XGBRegressor(random_state=42, objective='reg:squarederror')
rand_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_grid,
    n_iter=15,          # 전체 중 15조합 탐색
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 고정된 최적 파라미터로 학습
        model = XGBRegressor(**best_params, random_state=42, objective='reg:squarederror')
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE 계산
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 XGBoost 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'subsample': 0.8, 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 0.3, 'colsample_bytree': 1.0}

📊 XGBoost 롤링 예측 결과
📉 평균 Train MAE: 0.8190
📈 평균 Test MAE: 2.8706


## XGB 예측값

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운수업lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ y, X 설정
# -----------------------------
target = 'gdp'
exog_vars = [
'생산지수','항공통계']

y = df[target].dropna()
X = df[exog_vars].dropna()

df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ RandomizedSearchCV (초기 구간)
# -----------------------------
param_grid = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3]
}

first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

xgb = XGBRegressor(random_state=42, objective='reg:squarederror')
rand_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_grid,
    n_iter=15,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 + pred 저장
# -----------------------------
train_mae_list = []
test_mae_list = []
rolling_preds = []   # ⭐ 예측값 저장

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = XGBRegressor(**best_params, random_state=42, objective='reg:squarederror')
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(test_pred),
            'actual': float(y_test.values[0])
        })

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 XGBoost 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 XGBoost 예측값(pred) 시계열:")
print(pred_df)


✅ 최적 하이퍼파라미터: {'subsample': 0.8, 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 0.3, 'colsample_bytree': 1.0}

📊 XGBoost 롤링 예측 결과
📉 평균 Train MAE: 0.8190
📈 평균 Test MAE: 2.8706

📌 XGBoost 예측값(pred) 시계열:
            pred    actual
date                      
2021Q1  0.478345  7.685563
2021Q2  4.028786  2.869762
2021Q3  3.718907 -0.316051
2021Q4  3.224388  2.610384
2022Q1  5.318460  2.658985
2022Q2  3.327878  5.777034
2022Q3  3.130667  2.496486
2022Q4  0.765067  4.233273
2023Q1 -2.364481  0.323446
2023Q2  3.443310  9.975364
2023Q3  3.047094  2.870310
2023Q4  1.202009  3.453564
2024Q1 -0.264653 -1.827102
2024Q2  2.374603  3.401592
2024Q3  2.709634  2.824389
2024Q4  7.077948  0.650825
2025Q1 -9.322132 -2.883479
2025Q2  4.288784  2.063035


In [ ]:
level_2025Q1 = 26223.5

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 27348.169154303076


#MLP

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운수업lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','항공통계'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# 스케일링 (NN에서는 매우 중요)
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), index=y.index)

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 하이퍼파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'hidden_layer_sizes': [(32,), (64,), (32,16), (64,32), (128,64)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'lbfgs'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

# 첫 번째 윈도우로 하이퍼파라미터 탐색
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X_scaled.loc[start_period:first_train_end]
first_y_train = y_scaled.loc[start_period:first_train_end]

mlp = MLPRegressor(random_state=42, max_iter=1000)
rand_search = RandomizedSearchCV(
    mlp,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y_scaled.loc[train_start:train_end]
    y_test = y_scaled.loc[test_end:test_end]
    X_train = X_scaled.loc[train_start:train_end]
    X_test = X_scaled.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = MLPRegressor(**best_params, random_state=42, max_iter=1000)
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # 스케일링 복원
        train_pred_inv = scaler_y.inverse_transform(train_pred.reshape(-1, 1)).flatten()
        test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1)).flatten()
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1, 1)).flatten()
        y_test_inv = scaler_y.inverse_transform(y_test.values.reshape(-1, 1)).flatten()

        # MAE 계산
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 Neural Network (MLP) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'solver': 'adam', 'learning_rate_init': 0.001, 'hidden_layer_sizes': (64,), 'alpha': 0.01, 'activation': 'relu'}

📊 Neural Network (MLP) 롤링 예측 결과
📉 평균 Train MAE: 0.9596
📈 평균 Test MAE: 2.6015


## MLP 예측값

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운수업lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ y, X 설정
# -----------------------------
target = 'gdp'
exog_vars = [
'생산지수','항공통계']

df_model = df[[target] + exog_vars].dropna()

y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 스케일링 (정상 코드)
# -----------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(
    scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(),
    index=y.index
)

# -----------------------------
# 4️⃣ 롤링 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 5️⃣ 하이퍼파라미터 축소 (안정성)
# -----------------------------
param_grid = {
    'hidden_layer_sizes': [(32,), (64,), (32,16)],
    'activation': ['relu'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.001],
    'learning_rate_init': [0.001, 0.005]
}

first_train_end = start_period + (train_window - 1)
first_X_train = X_scaled.loc[start_period:first_train_end]
first_y_train = y_scaled.loc[start_period:first_train_end]

mlp = MLPRegressor(random_state=42, max_iter=2000)
rand_search = RandomizedSearchCV(
    mlp, param_grid, n_iter=5, cv=3,
    scoring='neg_mean_absolute_error', n_jobs=-1, random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 파라미터:", best_params)

# -----------------------------
# 6️⃣ 롤링 예측
# -----------------------------
train_mae_list = []
test_mae_list = []
rolling_preds = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    train_start = train_end - (train_window - 1)
    test_end = test_start

    X_train = X_scaled.loc[train_start:train_end]
    y_train = y_scaled.loc[train_start:train_end]
    X_test = X_scaled.loc[test_end:test_end]
    y_test = y_scaled.loc[test_end:test_end]

    if len(X_train) < train_window:
        continue

    try:
        model = MLPRegressor(
            **best_params,
            random_state=42,
            max_iter=2000
        )
        model.fit(X_train, y_train)

        # ---- Train 예측 ----
        train_pred_scaled = model.predict(X_train)
        train_pred = scaler_y.inverse_transform(train_pred_scaled.reshape(-1,1)).flatten()
        train_real = scaler_y.inverse_transform(y_train.values.reshape(-1,1)).flatten()
        train_mae = mean_absolute_error(train_real, train_pred)
        train_mae_list.append(train_mae)

        # ---- Test 예측 ----
        test_pred_scaled = model.predict(X_test)
        test_pred = scaler_y.inverse_transform(test_pred_scaled.reshape(-1,1)).flatten()
        test_real = scaler_y.inverse_transform(y_test.values.reshape(-1,1)).flatten()
        test_mae = mean_absolute_error(test_real, test_pred)
        test_mae_list.append(test_mae)

        # ---- pred 저장 ----
        rolling_preds.append({
            'date': test_end,
            'pred': float(test_pred[0]),
            'actual': float(test_real[0])
        })

    except Exception as e:
        print("❌ 오류 발생:", e)
        continue

# -----------------------------
# 7️⃣ 결과 출력
# -----------------------------
print("\n📊 Neural Network (MLP) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

if len(rolling_preds) > 0:
    pred_df = pd.DataFrame(rolling_preds).set_index('date')
    print(pred_df)
else:
    print("⚠ rolling_preds가 텅 빔 → 모델 학습 실패")


✅ 최적 파라미터: {'solver': 'adam', 'learning_rate_init': 0.001, 'hidden_layer_sizes': (64,), 'alpha': 0.001, 'activation': 'relu'}

📊 Neural Network (MLP) 롤링 예측 결과
📉 평균 Train MAE: 0.9211
📈 평균 Test MAE: 2.6671
            pred    actual
date                      
2021Q1 -0.876916  7.685563
2021Q2  4.168279  2.869762
2021Q3  4.277902 -0.316051
2021Q4  9.009774  2.610384
2022Q1  3.774048  2.658985
2022Q2  4.880298  5.777034
2022Q3  3.112214  2.496486
2022Q4  0.731415  4.233273
2023Q1  2.814309  0.323446
2023Q2  2.790517  9.975364
2023Q3  4.009399  2.870310
2023Q4  0.383259  3.453564
2024Q1 -0.479513 -1.827102
2024Q2  1.898123  3.401592
2024Q3  2.434101  2.824389
2024Q4  2.476753  0.650825
2025Q1 -1.085378 -2.883479
2025Q2  2.337165  2.063035


In [ ]:
level_2025Q1 = 26223.5

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 26836.3864773683


#LSTM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운수업lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','항공통계']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 스케일링
# -----------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), index=y.index)

# -----------------------------
# 4️⃣ LSTM 입력 형태 변환 함수
# -----------------------------
def create_lstm_input(X, y, lookback=4):
    """
    lookback: 과거 몇 분기 데이터를 사용할지 (ex. 4 = 1년)
    """
    Xs, ys, idxs = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X.iloc[i-lookback:i].values)
        ys.append(y.iloc[i])
        idxs.append(y.index[i])
    return np.array(Xs), np.array(ys), idxs

lookback = 4
X_seq, y_seq, idxs = create_lstm_input(X_scaled, y_scaled, lookback)
y_seq = pd.Series(y_seq, index=idxs)

# -----------------------------
# 5️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 6️⃣ LSTM 모델 정의 함수
# -----------------------------
def build_lstm(input_shape):
    model = Sequential()
    model.add(LSTM(64, activation='tanh', input_shape=input_shape, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mae')
    return model

# -----------------------------
# 7️⃣ 롤링 예측
# -----------------------------
train_mae_list, test_mae_list = [], []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    # 윈도우 범위 맞추기
    train_mask = (np.array(idxs) >= train_start) & (np.array(idxs) <= train_end)
    test_mask = (np.array(idxs) == test_end)

    X_train, y_train = X_seq[train_mask], y_seq[train_mask]
    X_test, y_test = X_seq[test_mask], y_seq[test_mask]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = build_lstm((X_train.shape[1], X_train.shape[2]))
        early_stop = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
        model.fit(X_train, y_train, epochs=200, batch_size=8, verbose=0, callbacks=[early_stop])

        # 예측
        train_pred = model.predict(X_train, verbose=0)
        test_pred = model.predict(X_test, verbose=0)

        # 스케일링 복원
        train_pred_inv = scaler_y.inverse_transform(train_pred)
        test_pred_inv = scaler_y.inverse_transform(test_pred)
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1, 1))
        y_test_inv = scaler_y.inverse_transform(y_test.values.reshape(-1, 1))

        # MAE 계산
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 8️⃣ 결과 요약
# -----------------------------
print("\n📊 LSTM 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")



📊 LSTM 롤링 예측 결과
📉 평균 Train MAE: 2.3254
📈 평균 Test MAE: 2.2952


## LSTM 예측값

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운수업lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ y, X 설정
# -----------------------------
target = 'gdp'
exog_vars = [
'생산지수','항공통계'
]

df_model = pd.concat([df[target], df[exog_vars]], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 스케일링
# -----------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(
    scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(),
    index=y.index
)

# -----------------------------
# 4️⃣ LSTM 입력 생성
# -----------------------------
def create_lstm_input(X, y, lookback=4):
    Xs, ys, idxs = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X.iloc[i-lookback:i].values)
        ys.append(y.iloc[i])
        idxs.append(y.index[i])
    return np.array(Xs), np.array(ys), idxs

lookback = 4
X_seq, y_seq, idxs = create_lstm_input(X_scaled, y_scaled, lookback)
y_seq = pd.Series(y_seq, index=idxs)

# -----------------------------
# 5️⃣ 롤링 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 6️⃣ 모델 생성 함수
# -----------------------------
def build_lstm(input_shape):
    model = Sequential()
    model.add(LSTM(64, activation='tanh', input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mae')
    return model

# -----------------------------
# 7️⃣ 롤링 예측 + pred 저장
# -----------------------------
train_mae_list, test_mae_list = [], []
rolling_preds = []  # ⭐ 예측값 저장 리스트

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    train_start = train_end - (train_window - 1)
    test_end = test_start

    # LSTM용 슬라이싱
    idx_arr = np.array(idxs)

    train_mask = (idx_arr >= train_start) & (idx_arr <= train_end)
    test_mask  = (idx_arr == test_end)

    X_train, y_train = X_seq[train_mask], y_seq[train_mask]
    X_test,  y_test  = X_seq[test_mask], y_seq[test_mask]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = build_lstm((X_train.shape[1], X_train.shape[2]))
        early = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)

        model.fit(X_train, y_train, epochs=200, batch_size=8,
                  verbose=0, callbacks=[early])

        # ---- Train 예측 ----
        train_pred = model.predict(X_train, verbose=0)
        test_pred  = model.predict(X_test, verbose=0)

        # ---- 스케일 복원 ----
        train_pred_inv = scaler_y.inverse_transform(train_pred)
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1,1))

        test_pred_inv = scaler_y.inverse_transform(test_pred)
        y_test_inv  = scaler_y.inverse_transform(y_test.values.reshape(-1,1))

        # ---- MAE ----
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(test_pred_inv[0]),
            'actual': float(y_test_inv[0])
        })

    except Exception as e:
        print(f"❌ Error: {e}")
        continue

# -----------------------------
# 8️⃣ 결과 요약 + pred_df 출력
# -----------------------------
print("\n📊 LSTM 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE:  {np.mean(test_mae_list):.4f}")

# ⭐ 예측값 시계열 출력
pred_df = pd.DataFrame(rolling_preds).set_index('date')
print("\n📌 LSTM 예측값(pred) 시계열:")
print(pred_df)



📊 LSTM 롤링 예측 결과
📉 평균 Train MAE: 2.2208
📈 평균 Test MAE:  2.3304

📌 LSTM 예측값(pred) 시계열:
            pred    actual
date                      
2022Q1  0.443079  2.658985
2022Q2  0.210177  5.777034
2022Q3  3.181190  2.496486
2022Q4  2.998697  4.233273
2023Q1  3.470533  0.323446
2023Q2  3.194077  9.975364
2023Q3  2.151119  2.870310
2023Q4  3.640451  3.453564
2024Q1  3.072165 -1.827102
2024Q2  3.511624  3.401592
2024Q3  2.781648  2.824389
2024Q4  2.348008  0.650825
2025Q1  1.996227 -2.883479
2025Q2  1.602562  2.063035


In [ ]:
level_2025Q1 = 26223.5

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 26643.747833137513


#단변량 아리마

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운수업lag.csv')

# 인덱스가 분기형일 경우 변환
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상 설정
# -----------------------------
y = df['gdp'].dropna()

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간
start_period = pd.Period('2021Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ (p,d,q) 탐색 범위
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                # 데이터 슬라이싱
                y_train = y.loc[train_start:train_end]
                y_test = y.loc[test_end:test_end]

                if len(y_train) < train_window:
                    continue

                try:
                    # -------------------------
                    # 모델 학습
                    # -------------------------
                    model = ARIMA(y_train, order=(p, d, q))
                    fit = model.fit()

                    # -------------------------
                    # 예측 및 MAE 계산
                    # -------------------------
                    pred = fit.forecast(steps=1)

                    train_mae = mean_absolute_error(y_train, fit.fittedvalues)
                    test_mae = mean_absolute_error(y_test, pred)

                    train_mae_list.append(train_mae)
                    test_mae_list.append(test_mae)

                except Exception as e:
                    print(f"❌ Error at (p,d,q)=({p},{d},{q}): {e}")
                    continue

            # -------------------------
            # 평균 MAE 저장
            # -------------------------
            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)

if results_df.empty:
    print("⚠️ 결과가 없습니다. 모든 (p,d,q) 조합이 실패했습니다.")
else:
    best = results_df.loc[results_df['test_MAE'].idxmin()]
    print("✅ 최적 ARIMA(p,d,q):", tuple(best[['p', 'd', 'q']]))
    print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
    print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

    display(results_df.sort_values('test_MAE').head(10))


✅ 최적 ARIMA(p,d,q): (0.0, 1.0, 1.0)
📉 평균 Train MAE: 3.3653
📈 평균 Test MAE: 2.6883


,p,d,q,train_MAE,test_MAE
1,0,1,1,3.365337,2.688321
13,1,1,1,3.299642,2.760803
14,1,1,2,3.252938,2.812180
2,0,1,2,3.305417,2.953207
6,0,2,2,4.053904,2.985942
15,1,1,3,3.292285,3.014344
19,1,2,3,3.914228,3.122359
25,2,1,1,3.314600,3.181277
3,0,1,3,3.251760,3.273359
12,1,1,0,3.665969,3.290614


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운수업qoq.csv')

# 인덱스가 분기형일 경우 변환
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상 설정
# -----------------------------
y = df['gdp'].dropna()

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간
start_period = pd.Period('2021Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ (p,d,q) 탐색 범위
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                # 데이터 슬라이싱
                y_train = y.loc[train_start:train_end]
                y_test = y.loc[test_end:test_end]

                if len(y_train) < train_window:
                    continue

                try:
                    # -------------------------
                    # 모델 학습
                    # -------------------------
                    model = ARIMA(y_train, order=(p, d, q))
                    fit = model.fit()

                    # -------------------------
                    # 예측 및 MAE 계산
                    # -------------------------
                    pred = fit.forecast(steps=1)

                    train_mae = mean_absolute_error(y_train, fit.fittedvalues)
                    test_mae = mean_absolute_error(y_test, pred)

                    train_mae_list.append(train_mae)
                    test_mae_list.append(test_mae)

                except Exception as e:
                    print(f"❌ Error at (p,d,q)=({p},{d},{q}): {e}")
                    continue

            # -------------------------
            # 평균 MAE 저장
            # -------------------------
            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)

if results_df.empty:
    print("⚠️ 결과가 없습니다. 모든 (p,d,q) 조합이 실패했습니다.")
else:
    best = results_df.loc[results_df['test_MAE'].idxmin()]
    print("✅ 최적 ARIMA(p,d,q):", tuple(best[['p', 'd', 'q']]))
    print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
    print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

    display(results_df.sort_values('test_MAE').head(10))


✅ 최적 ARIMA(p,d,q): (0.0, 1.0, 1.0)
📉 평균 Train MAE: 3.3653
📈 평균 Test MAE: 2.6883


,p,d,q,train_MAE,test_MAE
1,0,1,1,3.365337,2.688321
13,1,1,1,3.299642,2.760803
14,1,1,2,3.252938,2.812180
2,0,1,2,3.305417,2.953207
6,0,2,2,4.053904,2.985942
15,1,1,3,3.292285,3.014344
19,1,2,3,3.914228,3.122359
25,2,1,1,3.314600,3.181277
3,0,1,3,3.251760,3.273359
12,1,1,0,3.665969,3.290614


## 단변량 아리마 예측값

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운수업lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상
# -----------------------------
y = df['gdp'].dropna()

# -----------------------------
# 3️⃣ 롤링 윈도우
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2021Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ (p,d,q) 범위
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                y_train = y.loc[train_start:train_end]
                y_test = y.loc[test_end:test_end]

                if len(y_train) < train_window:
                    continue

                try:
                    model = ARIMA(y_train, order=(p, d, q))
                    fit = model.fit()

                    pred = fit.forecast(steps=1)

                    train_mae = mean_absolute_error(y_train, fit.fittedvalues)
                    test_mae = mean_absolute_error(y_test, pred)

                    train_mae_list.append(train_mae)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 최적 모델 선택
# -----------------------------
results_df = pd.DataFrame(results)

if results_df.empty:
    print("⚠️ 모든 조합 실패")
else:
    best = results_df.loc[results_df['test_MAE'].idxmin()]
    best_p, best_d, best_q = int(best['p']), int(best['d']), int(best['q'])

    print("✅ 최적 ARIMA(p,d,q):", (best_p, best_d, best_q))
    print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
    print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

# -----------------------------
# 8️⃣ ⭐ 최적 모델로 pred 저장 (재예측)
# -----------------------------
rolling_preds = []
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]

    if len(y_train) < train_window:
        continue

    try:
        model = ARIMA(y_train, order=(best_p, best_d, best_q))
        fit = model.fit()

        pred = fit.forecast(steps=1)

        # MAE
        train_mae_list.append(mean_absolute_error(y_train, fit.fittedvalues))
        test_mae_list.append(mean_absolute_error(y_test, pred))

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(pred),
            'actual': float(y_test.values[0])
        })

    except Exception as e:
        print("❌ 재예측 오류:", e)
        continue

# -----------------------------
# 9️⃣ pred_df 출력
# -----------------------------
pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 ARIMA 최적모델 예측(pred) 시계열:")
print(pred_df)

print("\n📉 최종 Train MAE:", np.mean(train_mae_list))
print("📈 최종 Test MAE:", np.mean(test_mae_list))


✅ 최적 ARIMA(p,d,q): (0, 1, 1)
📉 평균 Train MAE: 3.3653
📈 평균 Test MAE: 2.6883

📌 ARIMA 최적모델 예측(pred) 시계열:
            pred    actual
date                      
2021Q1 -0.447549  7.685563
2021Q2  0.220728  2.869762
2021Q3  0.309919 -0.316051
2021Q4  0.325916  2.610384
2022Q1  0.561653  2.658985
2022Q2  0.496237  5.777034
2022Q3  2.413526  2.496486
2022Q4  2.407527  4.233273
2023Q1  2.993302  0.323446
2023Q2  2.217760  9.975364
2023Q3  4.243299  2.870310
2023Q4  3.813710  3.453564
2024Q1  3.774248 -1.827102
2024Q2  2.475903  3.401592
2024Q3  2.671919  2.824389
2024Q4  2.696857  0.650825
2025Q1  1.639596 -2.883479
2025Q2  2.064179  2.063035

📉 최종 Train MAE: 3.3653370002407312
📈 최종 Test MAE: 2.6883208972733876


In [ ]:
level_2025Q1 = 26223.5

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 26764.799865434936
